In [5]:
import random
import copy

# Card
class Card:
    def __init__(self, color, number, is_skip=False):
        self.color = color
        self.number = number
        self.is_skip = is_skip

    def __str__(self):
        if self.is_skip:
            return f"{self.color} Skip"
        return f"{self.color} {self.number}"

def generate_deck():
    colors = ["Red", "Blue", "Green", "Yellow"]
    deck = []

    for color in colors:
        for num in range(10):
            deck.append(Card(color, num))

        # add skip cards
        deck.append(Card(color, -1, is_skip=True))

    random.shuffle(deck)
    return deck

def deal_cards(deck, num_players=3):
    hands = [[] for _ in range(num_players)]

    for _ in range(5):
        for i in range(num_players):
            hands[i].append(deck.pop())

    return hands

def get_valid_moves(hand, top_card):
    valid = []

    for card in hand:
        if card.color == top_card.color or card.number == top_card.number or card.is_skip:
            valid.append(card)

    return valid

def apply_move(state, player, move):
    new_state = copy.deepcopy(state)

    if move == "draw":
        if new_state["deck"]:
            new_state["hands"][player].append(new_state["deck"].pop())
        return new_state

    # FIX: remove card safely
    for c in new_state["hands"][player]:
        if c.color == move.color and c.number == move.number and c.is_skip == move.is_skip:
            new_state["hands"][player].remove(c)
            break

    new_state["top_card"] = move

    # handle skip
    if move.is_skip:
        new_state["skip"] = True
    else:
        new_state["skip"] = False

    return new_state

def evaluate(state, player, strategy="defensive"):
    my_cards = len(state["hands"][player])

    opp_cards = []
    for i in range(3):
        if i != player:
            opp_cards.append(len(state["hands"][i]))

    avg_opp = sum(opp_cards) / 2

    skip_cards = sum(1 for c in state["hands"][player] if c.is_skip)

    # baseline
    score = 50 - 5 * my_cards + 2 * avg_opp + 3 * skip_cards

    # tuning strategies
    if strategy == "defensive":
        score += 3 * skip_cards   # more value to skip
    else:
        score += 3 * (10 - my_cards)  # aggressive

    return score

def minimax(state, depth, player):
    if depth == 0 or is_terminal(state):
        return evaluate(state, player, "defensive")

    moves = get_valid_moves(state["hands"][player], state["top_card"])

    if not moves:
        moves = ["draw"]

    best = float("-inf")

    for move in moves:
        new_state = apply_move(state, player, move)
        value = minimax(new_state, depth - 1, player)
        best = max(best, value)

    return best

def expectimax(state, depth, player):
    if depth == 0 or is_terminal(state):
        return evaluate(state, player, "offensive")

    moves = get_valid_moves(state["hands"][player], state["top_card"])

    if not moves:
        # chance node (draw)
        if not state["deck"]:
            return evaluate(state, player, "offensive")

        total = 0
        prob = 1 / len(state["deck"])

        for card in state["deck"]:
            new_state = copy.deepcopy(state)
            new_state["hands"][player].append(card)
            total += prob * expectimax(new_state, depth - 1, player)

        return total

    best = float("-inf")

    for move in moves:
        new_state = apply_move(state, player, move)
        value = expectimax(new_state, depth - 1, player)
        best = max(best, value)

    return best

def is_terminal(state):
    for hand in state["hands"]:
        if len(hand) == 0:
            return True
    return False

def choose_move(state, player, mode):
    hand = state["hands"][player]
    top_card = state["top_card"]

    moves = get_valid_moves(hand, top_card)

    print("\nTop card:", top_card)
    print(f"\nPlayer {player + 1} hand:")
    
    for i, c in enumerate(hand):
        print(f"{i}: {c}")
        
    #Manual
    if player == 2 and mode == "manual":
        if not moves:
            print("No valid moves. Drawing card...")
            return "draw"

        print("\nChoose a move index:")
        for i, m in enumerate(moves):
            print(f"{i}: {m}")

        while True:
            try:
                choice = int(input("Enter index: "))
                if 0 <= choice < len(moves):
                    return moves[choice]
            except:
                pass
            print("Invalid input, try again.")

    #AI
    if not moves:
        return "draw"

    print("\nPossible moves:")

    best_move = None
    best_score = float("-inf")

    for move in moves:
        temp_state = apply_move(state, player, move)

        if player == 0:  # Player 1 (Minimax)
            score = minimax(temp_state, 2, player)

        elif player == 1:  # Player 2 (Expectimax)
            score = expectimax(temp_state, 2, player)

        else:  # Player 3 in simulation mode
            score = minimax(temp_state, 2, player)

        print(f"Play: {move} -> Score: {round(score,2)}")

        if score > best_score:
            best_score = score
            best_move = move

    return best_move

def play_game():
    mode = input("Choose mode for Player 3 (manual/sim): ").lower()

    deck = generate_deck()
    hands = deal_cards(deck)

    state = {
        "hands": hands,
        "top_card": deck.pop(),
        "deck": deck,
        "skip": False
    }

    current_player = 0

    while True:
        print("\n----------------------")
        print(f"Player {current_player + 1} Turn")

        if state["skip"]:
            print("Player skipped!")
            state["skip"] = False
            current_player = (current_player + 1) % 3
            continue

        move = choose_move(state, current_player, mode)

        print("Chosen Move:", move)

        state = apply_move(state, current_player, move)

        # win condition
        if len(state["hands"][current_player]) == 0:
            print(f"\nPlayer {current_player + 1} WINS!")
            break

        current_player = (current_player + 1) % 3

play_game()


Choose mode for Player 3 (manual/sim):  sim



----------------------
Player 1 Turn

Top card: Yellow 1

Player 1 hand:
0: Green 3
1: Blue 6
2: Red 5
3: Yellow 7
4: Blue 4

Possible moves:
Play: Yellow 7 -> Score: 40.0
Chosen Move: Yellow 7

----------------------
Player 2 Turn

Top card: Yellow 7

Player 2 hand:
0: Green 6
1: Blue 9
2: Blue 8
3: Yellow 6
4: Blue 0

Possible moves:
Play: Yellow 6 -> Score: 57.32
Chosen Move: Yellow 6

----------------------
Player 3 Turn

Top card: Yellow 6

Player 3 hand:
0: Yellow 3
1: Green 8
2: Red Skip
3: Green 1
4: Red 6

Possible moves:
Play: Yellow 3 -> Score: 48.0
Play: Red Skip -> Score: 38.0
Play: Red 6 -> Score: 38.0
Chosen Move: Yellow 3

----------------------
Player 1 Turn

Top card: Yellow 3

Player 1 hand:
0: Green 3
1: Blue 6
2: Red 5
3: Blue 4

Possible moves:
Play: Green 3 -> Score: 33.0
Chosen Move: Green 3

----------------------
Player 2 Turn

Top card: Green 3

Player 2 hand:
0: Green 6
1: Blue 9
2: Blue 8
3: Blue 0

Possible moves:
Play: Green 6 -> Score: 52.36
Chosen Move